# Week 3 – Cross-Cultural Market Basket Analysis

**Research question (option 01 – Event):** Does the same item pair behave differently during Ramadan than in baseline (non-event) transactions?

**Scope:** Istanbul (`TR_Istanbul`) and Dubai (`AE_Dubai`) only – the two markets where Ramadan occurs. Restricting both groups to the same markets keeps the market mix constant, so a difference is less likely to be a city effect.

**Context A:** `Cultural_Event == "Ramadan"`  
**Context B (baseline):** `Cultural_Event == "None"`

Other events in these markets (Eid) are excluded from both groups.

## 0. Setup

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import apriori, association_rules

SEED = 42  # Apriori is deterministic, but the README should state the seed
DATA_PATH = "Cross Cultural Market Basket Dataset v2.xlsx"

## 1. Load data
`keep_default_na=False` keeps `"None"` as a real category instead of NaN.  
Only the `Transactions` sheet is used (not `Instructor Notes`).

In [10]:
df = pd.read_excel(DATA_PATH, sheet_name="Transactions", keep_default_na=False)
print(df.shape)
df.head()

(900, 42)


,Transaction_ID,Market_ID,Country,Primary_Language,Region_Type,Urban_Density,Age_Group,Household_Type,Meal_Occasion,Social_Setting,...,Fruit,Dessert,Soft_Drink,Mineral_Water,Sandwich,Dumplings,Curry_Sauce,Pasta,Cheese,Soup
0,TX00001,KR_Seoul,South Korea,Korean,Suburban,Medium,35-49,Family,Dinner,Family,...,0,1,1,1,0,1,0,1,0,0
1,TX00002,KR_Seoul,South Korea,Korean,Metropolitan Core,Medium,35-49,Couple,Dinner,Couple,...,0,0,0,0,0,0,0,0,0,0
2,TX00003,KR_Seoul,South Korea,Korean,Metropolitan Core,High,25-34,Couple,Lunch,Group,...,0,0,0,0,0,0,0,0,0,0
3,TX00004,KR_Seoul,South Korea,Korean,Metropolitan Core,High,18-24,Single,Lunch,Solo,...,1,0,0,0,0,0,0,0,0,0
4,TX00005,KR_Seoul,South Korea,Korean,Metropolitan Core,High,25-34,Family,Lunch,Solo,...,0,1,0,0,0,0,0,0,0,0


In [11]:
# The 22 product columns run from 'Coffee' to 'Soup'
PRODUCTS = list(df.loc[:, "Coffee":"Soup"].columns)
print(len(PRODUCTS), PRODUCTS)

22 ['Coffee', 'Tea', 'Milk', 'Dates', 'Bread', 'Butter', 'Rice', 'Kimchi', 'Instant_Noodles', 'Eggs', 'Chicken', 'Yogurt', 'Fruit', 'Dessert', 'Soft_Drink', 'Mineral_Water', 'Sandwich', 'Dumplings', 'Curry_Sauce', 'Pasta', 'Cheese', 'Soup']


## 2. Explore the contexts
Check group sizes before choosing a research question. Small groups (N < ~50) give unstable lift.

In [12]:
CONTEXT_COLS = ["Cultural_Event", "Social_Setting", "Meal_Occasion",
                "Shopping_Channel", "Promotion_Exposure"]

for col in CONTEXT_COLS:
    print(f"--- {col} ---")
    print(df[col].value_counts(), "\n")

--- Cultural_Event ---
Cultural_Event
None              483
Ramadan            99
Eid                69
Chuseok            37
Holi               37
New Year           32
Christmas          31
Hanami             30
Bastille Day       30
Diwali             27
Lunar New Year     25
Name: count, dtype: int64 

--- Social_Setting ---
Social_Setting
Solo      346
Family    217
Couple    215
Group     122
Name: count, dtype: int64 

--- Meal_Occasion ---
Meal_Occasion
Dinner        256
Lunch         221
Breakfast     181
Afternoon     144
Late Night     98
Name: count, dtype: int64 

--- Shopping_Channel ---
Shopping_Channel
Supermarket          298
E-commerce           204
Local Market         165
Convenience Store    130
Food Delivery        103
Name: count, dtype: int64 

--- Promotion_Exposure ---
Promotion_Exposure
None              483
Coupon            162
Bundle            150
Loyalty Reward    105
Name: count, dtype: int64 



In [13]:
# Overall popularity of each item (= single-item support)
df[PRODUCTS].mean().sort_values(ascending=False)

Rice               0.398889
Mineral_Water      0.307778
Bread              0.295556
Chicken            0.277778
Tea                0.263333
Soft_Drink         0.247778
Coffee             0.228889
Milk               0.187778
Dessert            0.183333
Fruit              0.153333
Sandwich           0.142222
Eggs               0.141111
Instant_Noodles    0.128889
Butter             0.115556
Cheese             0.113333
Pasta              0.112222
Soup               0.110000
Curry_Sauce        0.093333
Yogurt             0.088889
Dates              0.086667
Dumplings          0.075556
Kimchi             0.064444
dtype: float64

**Choice:** Ramadan vs baseline in Istanbul & Dubai (see top of notebook).

---
## 3. Filter two groups
Step 1: keep only the two markets. Step 2: split by `Cultural_Event`.

In [ ]:
MARKETS = ["TR_Istanbul", "AE_Dubai"]

markets_df = df[df["Market_ID"].isin(MARKETS)]

group_a = markets_df[markets_df["Cultural_Event"] == "Ramadan"]  # Context A
group_b = markets_df[markets_df["Cultural_Event"] == "None"]     # Context B (baseline)

print(f"Markets subset: N = {len(markets_df)}")
print(f"Ramadan:        N = {len(group_a)}")
print(f"Baseline:       N = {len(group_b)}")

## 4. One-hot matrix
The product columns are already 0/1, so no TransactionEncoder is needed – just select them and cast to `bool`.

In [ ]:
basket_a = group_a[PRODUCTS].astype(bool)
basket_b = group_b[PRODUCTS].astype(bool)

print(basket_a.shape, basket_b.shape)

# Sanity check: item frequency (single-item support) side by side
item_support = pd.DataFrame({
    "Ramadan": basket_a.mean(),
    "Baseline": basket_b.mean(),
}).sort_values("Ramadan", ascending=False)
item_support.round(3)

## 5. Apriori – same thresholds for both groups
_Hint:_ Define `MIN_SUPPORT` and `MIN_CONFIDENCE` once as constants and use them for both groups.  
Look at `apriori(..., use_colnames=True)` and `association_rules(..., metric=..., min_threshold=...)`.

In [ ]:
MIN_SUPPORT = None     # TODO
MIN_CONFIDENCE = None  # TODO

## 6. Compare one rule A → B
Table: Context | N | Support | Confidence | Lift, plus a Δ row.  
_Hint:_ Write a small function that computes the measures manually from the 0/1 columns – then it works even if the rule falls below the threshold in one group.

In [ ]:
# TODO

## 7. Visualization
_Instructor's suggestion:_ number of rules per confidence threshold for each group. Or a bar chart of support/confidence/lift for your rule.

In [ ]:
# TODO

## 8. Interpretation (~300 words)
- Observed difference (with N)
- Alternative explanations: price, availability, promotion, channel, basket size, small sample
- No claims of cultural causation

## 9. BI action
- **Action:**
- **KPI:**
- **Guardrail:**

## 10. Limitations and data source
- Synthetic data, ...